In [3]:
# ============================================================
# PLOTTING-ONLY CELL
# Reads saved bootstrap products; NO bootstrap/data processing.
# ============================================================

from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe

import cartopy.crs as ccrs
import cartopy.feature as cfeature


# ============================================================
# PATHS
# ============================================================

PRODUCTS_PATH = Path(
    "../figures/mechanism_composites/"
    "mechanism_phase_vs_neutral_bootstrap_products.nc"
)

PNG_PATH = Path(
    "../figures/mechanism_composites/"
    "mechanism_phase_vs_neutral_bootstrap_16panel_redesign.png"
)

PDF_PATH = Path(
    "../figures/mechanism_composites/"
    "mechanism_phase_vs_neutral_bootstrap_16panel_redesign.pdf"
)


# ============================================================
# SETTINGS
# ============================================================

LON_MIN, LON_MAX = 29, 65
LAT_MIN, LAT_MAX = 5, 39

ROBUST_PCT = 98

QUIVER_STRIDE = 6

STIPPLE_STRIDE = 1
STIPPLE_SIZE = 1.6
STIPPLE_ALPHA = 0.62


ERL_RC = {
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 10,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 8,
    "path.simplify": False,
    "savefig.transparent": False,
}


COLUMN_KEYS = [
    "el_nino",
    "la_nina",
    "piod",
    "niod",
]

COLUMN_TITLES = [
    "El Niño",
    "La Niña",
    "pIOD",
    "nIOD",
]

PANEL_LABELS = [
    "(a)", "(b)", "(c)", "(d)",
    "(e)", "(f)", "(g)", "(h)",
    "(i)", "(j)", "(k)", "(l)",
    "(m)", "(n)", "(o)", "(p)",
]


# ============================================================
# LOAD SAVED PRODUCTS
# ============================================================

ds = xr.open_dataset(PRODUCTS_PATH).load()

lats = ds["latitude"].values
lons = ds["longitude"].values

terrain_mask = ds["terrain_925_mask"].values.astype(bool)


# ============================================================
# BUILD DICTIONARIES EXPECTED BY PLOT
# ============================================================

observed = {}
sig = {}

scalar_fields = ["z", "omega", "q", "mfmag"]
vector_fields = ["u", "v", "mfu", "mfv"]

for key in COLUMN_KEYS:

    observed[key] = {}

    for field in scalar_fields + vector_fields:
        observed[key][field] = ds[f"{key}_{field}"].values

    sig[key] = {}

    for field in scalar_fields:
        sig[key][field] = ds[f"{key}_{field}_sig"].values.astype(bool)


# ============================================================
# HELPERS
# ============================================================

def centered_levels_from_arrays(
    arrays,
    nlev=21,
    pct=ROBUST_PCT
):
    vals = []

    for a in arrays:
        x = np.asarray(a).ravel()
        x = x[np.isfinite(x)]

        if x.size:
            vals.append(x)

    if not vals:
        return np.linspace(-1, 1, nlev)

    vals = np.concatenate(vals)

    vmax = np.nanpercentile(
        np.abs(vals),
        pct
    )

    if not np.isfinite(vmax) or vmax == 0:
        vmax = np.nanmax(np.abs(vals))

    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    return np.linspace(
        -vmax,
        vmax,
        nlev
    )


def style_map(
    ax,
    show_grid_labels=False
):
    ax.set_extent(
        [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
        crs=ccrs.PlateCarree()
    )

    # Darker, more legible coastlines
    ax.coastlines(
        resolution="50m",
        linewidth=0.75,
        color="0.15",
        zorder=20
    )

    # Darker national borders
    ax.add_feature(
        cfeature.BORDERS.with_scale("50m"),
        linewidth=0.50,
        edgecolor="0.20",
        zorder=20
    )

    gl = ax.gridlines(
        draw_labels=show_grid_labels,
        linewidth=0.25,
        color="0.45",
        alpha=0.45,
        linestyle="--"
    )

    if show_grid_labels:
        gl.top_labels = False
        gl.right_labels = False

        gl.xlocator = mticker.FixedLocator(
            np.arange(30, 66, 5)
        )

        gl.ylocator = mticker.FixedLocator(
            np.arange(5, 40, 5)
        )

        gl.xlabel_style = {"size": 7}
        gl.ylabel_style = {"size": 7}


def add_panel_label(
    ax,
    label
):
    ax.text(
        0.02,
        0.98,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.80,
            pad=1.2
        ),
        zorder=30
    )


def add_stippling(
    ax,
    sig_mask,
    lons,
    lats,
    terrain_mask=None
):
    mask = np.asarray(
        sig_mask,
        dtype=bool
    ).copy()

    if terrain_mask is not None:
        mask &= ~terrain_mask

    if STIPPLE_STRIDE > 1:

        mask = mask[
            ::STIPPLE_STRIDE,
            ::STIPPLE_STRIDE
        ]

        lon_use = lons[::STIPPLE_STRIDE]
        lat_use = lats[::STIPPLE_STRIDE]

    else:

        lon_use = lons
        lat_use = lats

    yy, xx = np.where(mask)

    if yy.size == 0:
        return

    ax.scatter(
        lon_use[xx],
        lat_use[yy],
        s=STIPPLE_SIZE,
        c="black",
        marker=".",
        linewidths=0,
        alpha=STIPPLE_ALPHA,
        transform=ccrs.PlateCarree(),
        zorder=12
    )


def add_terrain_gray(
    ax,
    terrain_mask,
    lons,
    lats
):
    # MUCH lighter gray than original version
    ax.contourf(
        lons,
        lats,
        terrain_mask.astype(float),
        levels=[0.5, 1.5],
        colors=["0.88"],
        transform=ccrs.PlateCarree(),
        zorder=10
    )


def add_quiver(
    ax,
    u,
    v,
    lons,
    lats,
    terrain_mask=None,
    stride=QUIVER_STRIDE
):
    uplot = np.asarray(
        u,
        dtype=float
    ).copy()

    vplot = np.asarray(
        v,
        dtype=float
    ).copy()

    if terrain_mask is not None:
        uplot[terrain_mask] = np.nan
        vplot[terrain_mask] = np.nan

    u2 = uplot[
        ::stride,
        ::stride
    ]

    v2 = vplot[
        ::stride,
        ::stride
    ]

    lon2 = lons[::stride]
    lat2 = lats[::stride]

    lon2d, lat2d = np.meshgrid(
        lon2,
        lat2
    )

    return ax.quiver(
        lon2d,
        lat2d,
        u2,
        v2,
        transform=ccrs.PlateCarree(),
        scale=None,
        width=0.0022,
        headwidth=3.7,
        headlength=4.2,
        minlength=0.05,
        pivot="middle",
        color="black",
        path_effects=[
            pe.Stroke(
                linewidth=1.05,
                foreground="white"
            ),
            pe.Normal()
        ],
        zorder=15
    )


# ============================================================
# ROW DEFINITIONS
# ============================================================

row_specs = [
    (
        "z",
        "PuOr",
        "500 hPa $Z_g$",
        "m",
        None
    ),

    (
        "omega",
        "PuOr",
        "500 hPa $\\Omega$",
        "Pa s$^{-1}$",
        None
    ),

    (
        "q",
        "BrBG",
        "925 hPa $q$",
        "g kg$^{-1}$",
        ("u", "v")
    ),

    (
        "mfmag",
        "BrBG",
        r"925 hPa $q\mathbf{v}$",
        "g kg$^{-1}$ m s$^{-1}$",
        ("mfu", "mfv")
    ),
]


# ============================================================
# SHARED COLOR LEVELS
# ============================================================

levels = {}

for field, _, _, _, _ in row_specs:

    levels[field] = centered_levels_from_arrays(
        [
            observed[key][field]
            for key in COLUMN_KEYS
        ],
        nlev=21
    )


# ============================================================
# FIGURE
# ============================================================

proj = ccrs.PlateCarree()

with mpl.rc_context(ERL_RC):

    # Slightly wider figure -> larger panels
    fig, axes = plt.subplots(
        4,
        4,
        figsize=(11.5, 10.6),
        subplot_kw={"projection": proj},
        constrained_layout=False
    )

    # Less wasted white space between panels/rows
    fig.subplots_adjust(
        left=0.075,
        right=0.99,
        top=0.945,
        bottom=0.045,
        wspace=0.045,
        hspace=0.16
    )


    # --------------------------------------------------------
    # COLUMN HEADERS
    # --------------------------------------------------------

    for c, title in enumerate(COLUMN_TITLES):

        pos = axes[0, c].get_position()

        fig.text(
            (pos.x0 + pos.x1) / 2,
            0.975,
            title,
            ha="center",
            va="top",
            fontsize=10,
            fontweight="bold"
        )


    # --------------------------------------------------------
    # PANELS
    # --------------------------------------------------------

    panel_idx = 0
    mappables = {}

    for r, (
        field,
        cmap,
        row_label,
        cbar_label,
        vectors
    ) in enumerate(row_specs):

        is_925 = r >= 2

        for c, key in enumerate(COLUMN_KEYS):

            ax = axes[r, c]

            # Only left column gets coordinate labels
            style_map(
                ax,
                show_grid_labels=(c == 0)
            )

            plot_field = observed[key][field].copy()

            # Mask below-ground 925-hPa values
            if is_925:
                plot_field = np.where(
                    terrain_mask,
                    np.nan,
                    plot_field
                )

            cf = ax.contourf(
                lons,
                lats,
                plot_field,
                levels=levels[field],
                cmap=cmap,
                extend="both",
                transform=proj,
                zorder=1
            )

            if c == 0:
                mappables[field] = cf


            # -----------------------------------------------
            # SIGNIFICANCE
            # -----------------------------------------------

            add_stippling(
                ax,
                sig[key][field],
                lons,
                lats,
                terrain_mask=(
                    terrain_mask
                    if is_925
                    else None
                )
            )


            # -----------------------------------------------
            # VECTOR ROWS
            # -----------------------------------------------

            if vectors is not None:

                u_name, v_name = vectors

                qv = add_quiver(
                    ax,
                    observed[key][u_name],
                    observed[key][v_name],
                    lons,
                    lats,
                    terrain_mask=terrain_mask
                )

                # Keep only one vector key per row
                if c == 3:

                    if field == "q":

                        ax.quiverkey(
                            qv,
                            0.48,
                            -0.055,
                            5,
                            "5 m s$^{-1}$",
                            labelpos="E",
                            coordinates="axes",
                            fontproperties={
                                "size": 7
                            }
                        )

                    else:

                        ax.quiverkey(
                            qv,
                            0.42,
                            -0.055,
                            15,
                            "15 g kg$^{-1}$ m s$^{-1}$",
                            labelpos="E",
                            coordinates="axes",
                            fontproperties={
                                "size": 7
                            }
                        )


            # -----------------------------------------------
            # LIGHT TERRAIN OVERLAY
            # -----------------------------------------------

            if is_925:
                add_terrain_gray(
                    ax,
                    terrain_mask,
                    lons,
                    lats
                )


            # Panel label
            add_panel_label(
                ax,
                PANEL_LABELS[panel_idx]
            )

            panel_idx += 1


        # ----------------------------------------------------
        # ROW LABEL
        # Attach directly to the left-hand map so vertical
        # centering stays correct regardless of colorbar layout.
        # ----------------------------------------------------

        axes[r, 0].text(
            -0.27,
            0.50,
            row_label,
            transform=axes[r, 0].transAxes,
            ha="center",
            va="center",
            rotation=90,
            fontsize=8.5,
            fontweight="bold"
        )


        # ----------------------------------------------------
        # THINNER COLORBAR
        # ----------------------------------------------------

        cbar = fig.colorbar(
            mappables[field],
            ax=axes[r, :],
            orientation="horizontal",
            pad=0.045,
            shrink=0.84,
            fraction=0.025,
            aspect=45
        )

        cbar.set_label(
            cbar_label
        )

        cbar.ax.tick_params(
            labelsize=7
        )

        if field == "omega":
            cbar.ax.xaxis.set_major_formatter(
                mticker.FormatStrFormatter("%.2g")
            )


    # ========================================================
    # SAVE
    # ========================================================

    fig.savefig(
        PNG_PATH,
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        PDF_PATH,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02
    )

    plt.show()
    plt.close(fig)


print(f"Saved PNG: {PNG_PATH}")
print(f"Saved PDF: {PDF_PATH}")

ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'h5netcdf', 'scipy', 'cfgrib', 'gini', 'pynio', 'rasterio', 'zarr']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html